[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r2/r2-q2/03_sanity_checks_and_gradcam.ipynb)

# R2-Q2 Week 3 — Sanity-check Grad-CAM, then run it on the misclassification set

This notebook does two things in order. First, it runs Adebayo et al. (2018)'s sanity checks on vanilla Grad-CAM applied to the R2-Q1 classifier — the model-parameter randomization check and the data-randomization check — and reports the Spearman rank correlation between trained-model and randomized-model heatmaps at each randomization stage, with ρ < 0.3 at full randomization committed as the passing criterion. Second, once vanilla Grad-CAM has passed the sanity checks, it generates heatmaps for every image in the working set produced by Week 2 and saves them for Week 4's categorization work.

By the end of this notebook you will have:

- A `sanity_check_results.json` recording the Spearman ρ trajectory across randomization stages for both the model-parameter and data-randomization checks, the verdict against the ρ < 0.3 passing criterion, and the methods-section text justifying that the trained model's downstream interpretation rests on a sanity-checked saliency method.
- A `heatmaps/` directory containing one `.npy` file per misclassification — 81 files, each a 7×7 numpy array (pre-upsampling) — plus a `gradcam_metadata.parquet` index that joins the heatmap files back to the working set on `filename`. Week 4 upsamples each heatmap at use time.
- A `shuffled_resnet18.pt` checkpoint — the label-shuffled-trained classifier used for the data-randomization sanity check, saved so the check is reproducible without retraining.
- A diagnostic figure (`sanity_check_trajectory.png`) showing the Spearman ρ curve across randomization stages, so a reader can see the trajectory the verdict was made from, not just the verdict itself.

## Before you run anything: switch to a GPU runtime

This notebook trains a neural network, which needs a more powerful processing 
unit (GPU) to work. By default, Colab gives you a CPU-only runtime — fine for 
last week's notebooks, but it won't work for this one.

To switch:

1. In Colab's menu bar: **Runtime → Change runtime type**.
2. Under "Hardware accelerator," choose **T4 GPU**.
3. Click **Save**.

Wait for the session to come up on the new runtime (a few seconds).
Then run the cells below in order.

**If Colab tells you "no GPUs available right now":** free-tier GPU
access is shared and occasionally unavailable. Wait a few minutes
and try again. If it keeps refusing, tell your mentor — the
upgraded Colab tiers have more reliable GPU access.

In [ ]:
try:
    import irilab2026 as iri
    print(f"OK — irilab2026 {iri.__version__} is installed and all deps importable.")
    print("    → use Pattern 2 in the following Cell:")
    print("      !pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main")
except ModuleNotFoundError as e:
    print(f"MISSING — {e.name} not importable.")
    print("    → use Pattern 1 in the following Cell:")
    print("      !pip install git+https://github.com/geraldmc/irilab2026.git@main")

In [ ]:
# Pattern 1 — fresh or partial runtime (installs deps that aren't present yet)
# This skips anything pip already sees as installed. This is ideal for a new environment or when you want
# to be sure everything is up to date.

# !pip install git+https://github.com/geraldmc/irilab2026.git@main

# Pattern 2 — populated runtime (refreshes irilab2026 only, leaves deps alone)
# This is ideal for when you already have the dependencies installed and just want to update irilab2026.

# !pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main

### Confirm the GPU is in place

Before going further, confirm Colab attached a GPU. The cell below
prints `GPU detected:` followed by `True` or `False`. If it's `True`,
you're good. If it's `False`, go back to the top of the notebook and
re-do the runtime switch — the next cells will not work without a GPU.

The check is a separate cell from the main setup (below) because it's
fast and recoverable: if the GPU isn't there, you see the problem
right away with no Drive authorization prompt to dismiss first.

In [ ]:
import irilab2026 as iri

gpu_ok = iri.has_gpu()
print(f"GPU detected: {gpu_ok}")

assert gpu_ok, (
    "No GPU detected. Go to the top of the notebook and follow the "
    "runtime-switch instructions, then re-run from Setup Step 1."
)

In [ ]:
# Mount Drive, set up irilab2026 with a GPU requirement, seed everything,
# and point OUTPUT_DIR at the R2-Q1 output directory.
iri.setup(gpu_required=True)
iri.seed_all(42)

OUTPUT_DIR = iri.output_dir("r2_q1")
print(f"Output directory: {OUTPUT_DIR}")

# DEV_MODE switches the notebook between fast-iteration mode (tiny PV
# variant, 2 epochs — runs end to end in ~3 minutes on a T4) and the
# real production run (full PV variant, 10 epochs — ~25 minutes).
#
# Use DEV_MODE = True while you're debugging or making changes. Switch
# to False for the run that produces your real EQ#2 number.
DEV_MODE = True

if DEV_MODE:
    PV_VARIANT = "tiny"
    EPOCHS = 2
    print("DEV_MODE ON: PV variant = 'tiny', epochs = 2")
else:
    PV_VARIANT = "full"
    EPOCHS = 10
    print("DEV_MODE OFF: PV variant = 'full', epochs = 10")